# Acemi — 100 oyun: sekiz adımın öncesi / sonrası

**Kaggle → Internet açık → GITHUB_TOKEN secret → Run All.** Stockfish kurulmaz; normal doğrulama testleri/benchmark çalıştırılmaz. Yalnızca istenen 100 A/B oyunu oynanır.

- Yeni: `4068a0459f8119274014cd466270a2ffd3cd30c3` (sekiz adımın tamamı).
- Eski: `db058b344df8759810e957f11a58b4149055fe49` (sekiz adımdan önceki başlangıç).
- 50 farklı açılış × renk değişimli iki oyun = **100 oyun**. Her iki motora **100 ms/hamle**, Hash 16 MB, Threads 1, kitap kapalı. SPRT kapalıdır; erken başarı kararıyla durulmaz.
- Yeni sürüm `engine`, eski sürüm `opponent` rolündedir. W/D/L ve skor yeni sürüm açısından okunur. İki sürüm de zaman aşımı/geçersiz hamle/çökmede kaybeder.
- 10'ar oyunluk bölümler GitHub'a gönderilir. Tamamlanan bölümler tekrar oynatılmaz. Aynı SESSION_ID ile baştan çalıştırmak kayıtlı bölümlerden devam eder.
- Sadece Kaggle'da çalışır. Eski bir bölüm başladı fakat tamamlanmadıysa önce eski oturumu durdurun; sonra RETRY_INCOMPLETE=True ile devam edilebilir. Bitmiş oyun kayıtları değiştirilmez.


In [ ]:
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "codex/ab-100"
WORKSPACE = "/kaggle/working/acemi"
SESSION_ID = "eight-steps-100ms-100games"
GAMES = 100
CHUNK_GAMES = 10
MOVETIME_MS = 100
HASH_MB = 16
CONCURRENCY = 2
MAX_PLIES = 400
SEED = 1
RETRY_INCOMPLETE = False  # Set True only after stopping an interrupted old session.
OLD_COMMIT = "db058b344df8759810e957f11a58b4149055fe49"
NEW_COMMIT = "4068a0459f8119274014cd466270a2ffd3cd30c3"


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("This notebook runs matches only in Kaggle.")

def checked(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {args[0]}\n{(result.stdout + result.stderr)[-4000:]}")
    return result.stdout.strip()

workspace = Path(WORKSPACE)
if not workspace.exists():
    checked(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(workspace)])
else:
    checked(["git", "-C", str(workspace), "fetch", "origin", GITHUB_BRANCH])
    local = subprocess.run(["git", "-C", str(workspace), "show-ref", "--verify", "--quiet", f"refs/heads/{GITHUB_BRANCH}"])
    if local.returncode == 0:
        checked(["git", "-C", str(workspace), "switch", GITHUB_BRANCH])
    else:
        checked(["git", "-C", str(workspace), "switch", "--create", GITHUB_BRANCH, "--track", f"origin/{GITHUB_BRANCH}"])
    checked(["git", "-C", str(workspace), "merge", "--ff-only", f"origin/{GITHUB_BRANCH}"])
os.chdir(workspace)
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))
if not (workspace / "tools" / "ab_compare.py").exists():
    raise RuntimeError("Import the latest notebook from codex/ab-100 and rerun setup.")
checked([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"], cwd=workspace)

def pinned_checkout(path, commit):
    path = Path(path)
    if not path.exists():
        checked(["git", "clone", "--depth", "1", "--no-checkout", GITHUB_REPO_URL, str(path)])
    checked(["git", "-C", str(path), "fetch", "--depth", "1", "origin", commit])
    checked(["git", "-C", str(path), "checkout", "--detach", commit])
    if checked(["git", "-C", str(path), "rev-parse", "HEAD"]) != commit:
        raise RuntimeError("Pinned source mismatch")
    return path

new_source = pinned_checkout("/kaggle/working/acemi-ab-new", NEW_COMMIT)
old_source = pinned_checkout("/kaggle/working/acemi-ab-old", OLD_COMMIT)
session = workspace / "data" / "ab" / SESSION_ID
print("NEW:", NEW_COMMIT, "OLD:", OLD_COMMIT)
print("100 games; 50 paired openings; 100 ms/move; results:", session)


In [ ]:
# Publish records only. The token is never written to a file or a remote URL.
import base64
from kaggle_secrets import UserSecretsClient

def publish_results(message):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Add GITHUB_TOKEN under Kaggle Add-ons / Secrets.")
    auth = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": auth, "GIT_TERMINAL_PROMPT": "0"})
    def git(*args, check=True):
        result = subprocess.run(["git", *args], cwd=workspace, env=env, text=True, capture_output=True)
        if check and result.returncode:
            details = (result.stdout + result.stderr).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
            raise RuntimeError(f"Git {args[0]} failed: {details[-2000:]}")
        return result
    if git("branch", "--show-current").stdout.strip() != GITHUB_BRANCH:
        raise RuntimeError("Wrong results branch; rerun repository setup.")
    git("config", "user.name", "Acemi Kaggle A-B")
    git("config", "user.email", "acemi-kaggle-runner@users.noreply.github.com")
    paths = [str(session.relative_to(workspace))]
    paths += [p for p in ("data/runs", "data/index.csv", "data/positions") if (workspace / p).exists()]
    git("add", "--", *paths)
    if git("diff", "--cached", "--quiet", check=False).returncode:
        git("commit", "-m", message)
    git("fetch", "origin", GITHUB_BRANCH)
    if int(git("rev-list", "--count", "HEAD..FETCH_HEAD").stdout.strip()):
        git("rebase", "FETCH_HEAD")
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    print("Published:", message)


In [ ]:
# Create one immutable 100-game protocol, or resume this exact experiment.
from tools import ab_compare as ab
if Path(ab.__file__).resolve() != workspace / "tools" / "ab_compare.py":
    raise RuntimeError("Wrong tools package loaded; restart the Kaggle kernel and Run All.")

if (session / "protocol.json").exists():
    plan = ab.load_plan(session)
    expected = {"games": GAMES, "movetime_ms": MOVETIME_MS, "hash_mb": HASH_MB,
                "max_plies": MAX_PLIES, "selection_seed": SEED, "chunk_games": CHUNK_GAMES,
                "concurrency": min(CONCURRENCY, os.cpu_count() or 1)}
    if any(plan[key] != value for key, value in expected.items()):
        raise RuntimeError("Settings differ from saved experiment; use a new SESSION_ID.")
else:
    plan = ab.prepare(session, new_source, old_source, games=GAMES, chunk_games=CHUNK_GAMES,
                      movetime_ms=MOVETIME_MS, concurrency=CONCURRENCY, hash_mb=HASH_MB,
                      max_plies=MAX_PLIES, seed=SEED)
print((session / "SUMMARY.md").read_text())
publish_results(f"data: prepare 100-game A/B ({SESSION_ID})")


In [ ]:
# Exactly 10 chunks of 10 games. Completed chunks are verified and skipped.
for chunk in plan["chunks"]:
    command = [sys.executable, "-u", "-m", "tools.ab_compare", "run", "--session", str(session),
               "--chunk", str(chunk["number"])]
    if RETRY_INCOMPLETE:
        command.append("--retry-incomplete")
    print(f"\nA/B chunk {chunk['number']}/{len(plan['chunks'])}: {chunk['games']} games", flush=True)
    result = subprocess.run(command, cwd=workspace)
    try:
        publish_results(f"data: A/B chunk {chunk['number']} ({SESSION_ID})")
    except Exception:
        print("Push failed. Records remain in Kaggle. Retry the final cell before continuing.")
        raise
    if result.returncode:
        raise RuntimeError(f"A/B chunk stopped, exit={result.returncode}. Inspect the saved chunk log.")
    print((session / "SUMMARY.md").read_text())


In [ ]:
# Safe to rerun after a push failure: it does not start any games.
publish_results(f"data: finalize A/B records ({SESSION_ID})")
if (session / "results.json").exists():
    print((session / "SUMMARY.md").read_text())
    result = json.loads((session / "results.json").read_text())
    if result["recorded_attempts"] == GAMES:
        print("100-game A/B finished. W/D/L is from the NEW version's perspective.")
